# Aula 07 · Arquivos de texto e CSV

Este é o caderno da aula de hoje. Ele vem **sem as saídas** de propósito:
em cada exemplo você escreve antes o que acha que vai acontecer, e só
depois roda a célula. A previsão errada é a parte que ensina — não a apague.

**Ao fim desta aula você deve conseguir:**

1. ler um arquivo linha a linha com `with`, limpar cada linha e pular as vazias;
2. escolher entre `"w"` e `"a"` sabendo o que cada um faz com o conteúdo anterior;
3. ler um CSV com `DictReader`, guardar os registros numa lista e converter os campos antes de calcular.

## Preparação

Rode esta célula primeiro.

In [ ]:
# Os exemplos leem e escrevem arquivos. Para o roteiro poder ser conferido em
# qualquer maquina sem sujar o repositorio, tudo acontece numa pasta temporaria.
import csv
import os
import tempfile

pasta = tempfile.TemporaryDirectory()
os.chdir(pasta.name)

with open("alarmes.log", "w", encoding="utf-8") as arquivo:
    arquivo.write(
        "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal\n"
        "2026-03-02 09:00:00 INFO SWITCH-NORTE-02 porta ativada\n"
        "2026-03-02 10:02:55 CRITICAL OLT-CENTRO-01 temperatura alta\n"
        "\n"
    )

with open("medicoes.csv", "w", encoding="utf-8") as arquivo:
    arquivo.write(
        "equipamento,potencia_dbm,estado\n"
        "OLT-CENTRO-01,-21.4,UP\n"
        "ONU-SUL-4512,-27.0,UP\n"
        "OLT-NORTE-02,-19.8,DOWN\n"
    )

## Parte 1 — Exemplos: prever, rodar, investigar

### Q1. A linha vem com o `\n` — e existe uma linha em branco no fim

**Preveja:** "o arquivo tem 3 alarmes. Quantas linhas o laço vai imprimir?"

_Sua previsão:_

In [ ]:
with open("alarmes.log", encoding="utf-8") as arquivo:
    for linha in arquivo:
        print(repr(linha))

<details>
<summary><b>▶ O que aconteceu</b></summary>

Percorrer o arquivo com `for` entrega **uma linha por vez**, sem carregar tudo na
memória — é o que permite processar um log de milhares de linhas. Duas coisas
aparecem aqui: cada linha vem com o `\n` no fim (é o `repr` que o torna visível), e
o arquivo termina com uma **linha em branco**, porque todo arquivo de texto bem
formado termina com uma quebra. São três alarmes e quatro linhas.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#abrir-ler-fechar)

</details>

### Q2. O par de linhas que abre todo laço de leitura

**Preveja:** "agora quantas?"

_Sua previsão:_

In [ ]:
total = 0
with open("alarmes.log", encoding="utf-8") as arquivo:
    for linha in arquivo:
        linha = linha.strip()
        if len(linha) == 0:
            continue
        total = total + 1
print(total)

<details>
<summary><b>▶ O que aconteceu</b></summary>

Estas duas linhas abrem todo laço de leitura do curso: `linha.strip()` tira o `\n` e
os espaços das pontas, e `if len(linha) == 0: continue` pula a linha vazia. Sem
elas, um `linha.split()[2]` levanta `IndexError` na última volta — e essa é a causa
número um de erro em script de log. O `with`, por sua vez, fecha o arquivo sozinho
ao sair do bloco, mesmo se der erro no meio.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#limpando-e-contando)

</details>

### Q3. `"w"` apaga, `"a"` acrescenta

**Preveja:** "o que tem no arquivo depois das duas primeiras aberturas?"

_Sua previsão:_

In [ ]:
with open("relatorio.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write("primeira\n")
with open("relatorio.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write("segunda\n")
print(open("relatorio.txt", encoding="utf-8").read().splitlines())

with open("relatorio.txt", "a", encoding="utf-8") as arquivo:
    arquivo.write("terceira\n")
print(open("relatorio.txt", encoding="utf-8").read().splitlines())

<details>
<summary><b>▶ O que aconteceu</b></summary>

`"w"` **apaga** o conteúdo anterior no momento em que o arquivo é aberto, mesmo que
nada seja escrito depois; `"a"` acrescenta no fim, preservando o que havia. O `"w"`
é o que se quer no relatório que se regenera toda vez — e é desastroso quando o
nome do arquivo de saída coincide com o da entrada. E `write` **não** põe quebra de
linha: o `+ "\n"` é seu.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#escrever)

</details>

### Q4. `DictReader`: uma linha por vez, já como dicionário

**Preveja:** "o arquivo tem 4 linhas. Quantas o laço vai imprimir? E o cabeçalho, aparece?"

_Sua previsão:_

In [ ]:
with open("medicoes.csv", encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        print(registro)

<details>
<summary><b>▶ O que aconteceu</b></summary>

`csv.DictReader` faz três coisas de uma vez: lê a primeira linha como **cabeçalho**
e não a devolve como dado, transforma cada linha seguinte num **dicionário** usando
esse cabeçalho como chave, e entrega um registro por vez. Por isso saem três
registros de um arquivo de quatro linhas. O acesso é por nome de coluna, que
continua certo quando alguém acrescenta uma coluna no meio do arquivo.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#csv)

</details>

### Q5. Guardando numa lista, e a Unidade 2 volta a valer

**Preveja:** "que tipo de coisa é `medicoes` depois do primeiro laço?"

_Sua previsão:_

In [ ]:
medicoes = []
with open("medicoes.csv", encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        medicoes.append(registro)

print(len(medicoes))

no_ar = []
for m in medicoes:
    if m["estado"] == "UP":
        no_ar.append(m["equipamento"])
print(no_ar)

<details>
<summary><b>▶ O que aconteceu</b></summary>

O `DictReader` só funciona com o arquivo aberto e passa **uma única vez** pelas
linhas. Para usar os dados depois — percorrer duas vezes, calcular e depois filtrar
— acumule numa lista, que é o padrão de acumulação da Unidade 1. E repare no que
você tem no fim: uma **lista de dicionários**, exatamente o formato da Aula 06.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#guardando-as-linhas-para-usar-depois)

</details>

### Q5b. Investigue: percorrer duas vezes

**Preveja:** "o segundo laço sobre o mesmo `DictReader` conta quantos registros?"

_Sua previsão:_

In [ ]:
print(len(medicoes), len(medicoes))

with open("medicoes.csv", encoding="utf-8", newline="") as arquivo:
    leitor = csv.DictReader(arquivo)
    primeira_passada = 0
    for registro in leitor:
        primeira_passada = primeira_passada + 1
    segunda_passada = 0
    for registro in leitor:
        segunda_passada = segunda_passada + 1
print(primeira_passada, segunda_passada)

<details>
<summary><b>▶ O que aconteceu</b></summary>

conta **zero**. O `DictReader` é um leitor que avança: quando o
primeiro laço chega ao fim do arquivo, não sobra nada para o segundo. A lista, ao
contrário, responde `3` quantas vezes você perguntar. É exatamente por isso que se
acumula numa lista antes de usar os dados mais de uma vez.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#guardando-as-linhas-para-usar-depois)

</details>

### Q6. Tudo que vem do arquivo é texto

**Preveja:** "a segunda linha imprime `-21.4` ou outra coisa?"

_Sua previsão:_

In [ ]:
primeiro = medicoes[0]["potencia_dbm"]
print(repr(primeiro))
print(primeiro + "0")
print(float(primeiro) + 0)

<details>
<summary><b>▶ O que aconteceu</b></summary>

Tudo que vem de um arquivo é **texto**, inclusive o que parece número.
`medicoes[0]["potencia_dbm"]` é a string `"-21.4"`, e por isso `+ "0"` colou um
zero no fim em vez de somar. Só `float(...)` transforma aquilo num número. É o mesmo
`TypeError` da Aula 01, agora com dado de verdade.

[Mais sobre isto no capítulo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/#o-csv-vira-o-que-voce-ja-sabe-processar)

</details>

_Anotações da Parte 1:_

## Parte 2 — Resolver junto

Tente sozinho primeiro, por cinco minutos, na sua máquina. Depois
resolvemos juntos.

### E1. Ler e limpar

Escreva `linhas_uteis(caminho)`, que abre o arquivo, devolve a lista das linhas
**sem a quebra de linha** e **sem as linhas em branco**.

```python
linhas_uteis("alarmes.log")[0]
# -> "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3"
len(linhas_uteis("alarmes.log"))   # -> 5
```

O arquivo tem seis linhas — a última é em branco, como em todo arquivo de texto
bem formado. Ela não deve entrar.

**Assinatura:**

```python
def linhas_uteis(caminho):
    """Linhas do arquivo, sem quebra de linha e sem as vazias."""
```

In [ ]:
# sua solução aqui


### E2. Contar críticos no arquivo

Escreva `conta_criticos(caminho)`, que devolve quantos alarmes de severidade
`CRITICAL` há no arquivo de log.

```python
conta_criticos("alarmes.log")  # -> 3
```

Cuidado com a linha em branco do fim: ela não tem campo nenhum, e tentar acessar
`campos[2]` nela levanta `IndexError`.

**Assinatura:**

```python
def conta_criticos(caminho):
    """Quantos alarmes CRITICAL há no arquivo."""
```

In [ ]:
# sua solução aqui


### E3. Gravar o relatório

Escreva `salva_relatorio(caminho, linhas)`, que grava cada item da lista numa
linha do arquivo e devolve **quantas linhas** gravou. O arquivo é sobrescrito a
cada chamada.

```python
salva_relatorio("saida.txt", ["OLT-A  2", "ONU-B  1"])   # -> 2
```

Lembre que `write` não acrescenta a quebra de linha — ela é sua.

**Assinatura:**

```python
def salva_relatorio(caminho, linhas):
    """Grava uma linha por item e devolve quantas foram gravadas."""
```

In [ ]:
# sua solução aqui


### E4. Média do CSV

Escreva `media_potencia(caminho)`, que devolve a média das potências do CSV,
arredondada para duas casas. Arquivo sem nenhuma medição devolve `0.0`.

```python
media_potencia("medicoes.csv")  # -> -24.07
```

Lembre: o campo lido do CSV é **texto**. A conversão é sua.

**Assinatura:**

```python
import csv

def media_potencia(caminho):
    """Média das potências do CSV, com duas casas. Arquivo vazio devolve 0.0."""
```

In [ ]:
# sua solução aqui


_Anotações da Parte 2:_

## Parte 3 — Quiz de conceitos

Responda de cabeça, sem rodar. Conferimos juntos no fim.

**1.** Um arquivo com 3 alarmes, gravado normalmente, tem quantas linhas percorridas pelo `for`?

a) 3  b) 4  c) 2  d) depende do sistema

**2.** `with open(caminho) as arquivo:` tem a vantagem de:

a) ser mais rápido  b) fechar o arquivo sozinho ao sair do bloco  c) ler o arquivo inteiro  d) converter os campos

**3.** Abrir com `"w"` um arquivo que já existe:

a) acrescenta no fim  b) dá erro  c) apaga o conteúdo anterior  d) cria um arquivo novo com outro nome

**4.** `csv.DictReader` usa como chaves:

a) os números das colunas  b) a primeira linha do arquivo  c) os nomes que você passar  d) as letras A, B, C

**5.** `registro["potencia_dbm"] + 1`, com o CSV lido normalmente:

a) soma 1 à potência  b) dá `TypeError`  c) concatena `"1"` no fim  d) devolve `None`